In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [17]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix= stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [18]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [20]:
X, Y

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
          1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [ ]:
# embedding lookup table C:
# in the paper: 17K words in 30 dim spqce, Here as we have only 27, ok to start with a 2 dim embedding


In [21]:
C = torch.randn((27, 2))

In [23]:
C

tensor([[-0.0261,  0.8148],
        [ 0.9249,  0.5281],
        [-0.2078,  0.5555],
        [ 1.7961,  1.8419],
        [-0.1946,  0.0130],
        [ 0.0865,  1.6616],
        [-0.4026,  1.4530],
        [-0.0779, -0.4773],
        [ 0.2206, -0.3968],
        [ 1.2898,  0.4652],
        [-0.0048, -0.4839],
        [-0.6401,  1.0450],
        [ 0.4949,  0.9084],
        [-0.6014,  1.4995],
        [-0.3658,  0.2426],
        [ 0.6220, -0.2713],
        [-0.0629, -1.6374],
        [-0.1054, -0.1888],
        [-1.4285,  0.0168],
        [-1.1166, -1.0940],
        [ 0.1538, -1.5926],
        [ 0.9553, -0.3261],
        [-0.3351,  0.1436],
        [-0.6428, -1.1453],
        [ 0.8212,  1.8749],
        [ 0.6095,  0.8392],
        [-0.1775, -1.1901]])

In [35]:
C.shape

torch.Size([27, 2])

In [22]:
# before embedding all integers inside the input X, we make an example with 5
# one way is to index 5 in the lookup table C, getting the 5th row
C[5]

tensor([0.0865, 1.6616])

In [29]:
# the other way is to use one-hot encoding. Output is identical. All zero masking out all the rows but the 5th
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.0865, 1.6616])

In [ ]:
 # using indexing to do embedding is equal to use one-hot encoding. Here we use index as it is much faster.
# this can be seen as first layer of NN.

In [31]:
emb = C[X]

In [34]:
emb.shape

torch.Size([32, 3, 2])

In [33]:
X, C, emb

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([[-0.0261,  0.8148],
         [ 0.9249,  0.5281],
         [-0.2078,  0.5555],
         [ 1.7961,  1.8419],
         [-0.1946,  0.0130],
         [ 0.0865,  1.6616],
         [-0.4026,  1.4530],
         [-0.0779, -0.4773],
         [ 0.2206, -0.3968],
 

In [36]:
emb

tensor([[[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.0865,  1.6616]],

        [[-0.0261,  0.8148],
         [ 0.0865,  1.6616],
         [-0.6014,  1.4995]],

        [[ 0.0865,  1.6616],
         [-0.6014,  1.4995],
         [-0.6014,  1.4995]],

        [[-0.6014,  1.4995],
         [-0.6014,  1.4995],
         [ 0.9249,  0.5281]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [-0.0261,  0.8148]],

        [[-0.0261,  0.8148],
         [-0.0261,  0.8148],
         [ 0.6220, -0.2713]],

        [[-0.0261,  0.8148],
         [ 0.6220, -0.2713],
         [ 0.4949,  0.9084]],

        [[ 0.6220, -0.2713],
         [ 0.4949,  0.9084],
         [ 1.2898,  0.4652]],

        [[ 0.4949,  0.9084],
         [ 1.2898,  0.4652],
         [-0.3351,  0.1436]],

        [[ 1.2898,  0.4652],
         [-0.3351,  0.1436],
         [ 1.2898,  0.4652]],

        [[-0.3351,  0